In [2]:
import os
import glob

In [3]:
hosting_folder = r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\logs\hosting.6681806.experts"

# Get all files in the hosting folder
files = glob.glob(os.path.join(hosting_folder, "*"))

# Filter for files only (exclude directories)
files = [f for f in files if os.path.isfile(f)]

# Get the last modified file
if files:
    last_file = max(files, key=os.path.getmtime)

In [42]:
import pandas as pd
import re

def extract_strategy_data(file_path):
    """
    Extract strategy data from log file and return as DataFrame
    """
    data = []
    
    try:
        with open(file_path, 'r', encoding='UTF-16LE') as file:
            for line in file:
                # Skip empty lines
                if not line.strip():
                    continue
                    
                # Split by tabs
                parts = line.strip().split('\t')
                
                if len(parts) >= 5:
                    # Extract basic info
                    strategy_id = parts[0]
                    timestamp = parts[2]
                    
                    # Extract strategy name and parameters
                    strategy_info = parts[3]
                    
                    # Extract details from the last part
                    details = parts[4]
                    
                    # Parse details using regex
                    delta_match = re.search(r'Delta: ([\d.-]+)', details)
                    pnl_match = re.search(r'PnL: ([\d.-]+)', details)
                    trading_match = re.search(r'isTrading: (\w+)', details)
                    tp_match = re.search(r'TP: ([\d.-]+)', details)
                    sl_match = re.search(r'SL: ([\d.-]+)', details)
                    
                    # Extract symbol and magic if present
                    strategy_match = re.search(r'\[.*]', details)
                    symbol_match = re.search(r'Symbol: (\w+)', details)
                    magic_match = re.search(r'Magic: (\d+)', details)
                    position_match = re.search(r'Position: ([\d.-]+)', details)
                    volume_match = re.search(r'Volume: ([\d.-]+)', details)
                    valid_match = re.search(r'isValid: (\w+)', details)
                    
                    row = {
                        'Strategy_ID': strategy_match.group(0) if strategy_match else None,
                        'Timestamp': timestamp,
                        'Delta': delta_match.group(1) if delta_match else None,
                        'PnL': pnl_match.group(1) if pnl_match else None,
                        'isTrading': trading_match.group(1) if trading_match else None,
                        'TP': tp_match.group(1) if tp_match else None,
                        'SL': sl_match.group(1) if sl_match else None,
                        'Symbol': symbol_match.group(1) if symbol_match else None,
                        'Magic': magic_match.group(1) if magic_match else None,
                        'Position': position_match.group(1) if position_match else None,
                        'Volume': volume_match.group(1) if volume_match else None,
                        'isValid': valid_match.group(1) if valid_match else None
                    }
                    
                    data.append(row)
    
    except Exception as e:
        print(f"Error reading file: {e}")
        return None
    
    return pd.DataFrame(data)

df = extract_strategy_data(last_file)
df = df.groupby(['Strategy_ID', 'Timestamp']).last().reset_index()

In [43]:
df[df['isTrading'] == 'true'].groupby('Strategy_ID').last()

,Timestamp,Delta,PnL,isTrading,TP,SL,Symbol,Magic,Position,Volume,isValid
Strategy_ID,,,,,,,,,,,
[MAMA_DIST_18183975952241319922],15:20:37.694,4.00,-281.00,true,1650.00,-300.00,WINQ25,2782,4.00,4.00,true
[MAMA_DIST_2158147365549641725],16:42:29.911,5.00,-90.00,true,1700.00,-100.00,WINQ25,7252,5.00,5.00,true
[MAMA_DIST_2632308887531990214],15:31:57.265,3.00,-88.00,true,1700.00,-100.00,WINQ25,15089,3.00,3.00,true
[MOM_13227383262541241523],18:10:41.981,4.00,445.00,true,750.00,-700.00,WINQ25,3064,4.00,4.00,true
[MOM_357345436396352158],18:10:43.981,5.00,555.00,true,1550.00,-500.00,WINQ25,1654,5.00,5.00,true
[MOM_3869111060866295854],18:10:43.981,5.00,555.00,true,1800.00,-400.00,WINQ25,4237,5.00,5.00,true
[PRICES_MOM_14591597177401896865],18:10:43.981,5.00,1063.00,true,2000.00,-1600.00,WINQ25,26853,5.00,5.00,true
[P_RSI_16676336069753269574],18:10:43.981,4.00,318.00,true,700.00,-1850.00,WINQ25,16720,4.00,4.00,true
[P_RSI_RATIO_10378802357062804536],18:10:43.981,5.00,352.00,true,850.00,-1550.00,WINQ25,332,5.00,5.00,true
